In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os

In [ ]:
# Common path setup for local + Colab

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    REPO_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/masters-project")
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT.name != "masters-project" and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent

    if REPO_ROOT.name != "masters-project":
        raise RuntimeError("Could not find repo root folder named 'masters-project'.")

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
REPORTS_EVAL = REPO_ROOT / "reports" / "eval" / "modserve"
FIG_DIR = REPORTS_EVAL / "figures"
TABLE_DIR = REPORTS_EVAL / "tables"
CACHE_DIR = REPORTS_EVAL / "cache"
LOG_DIR = REPORTS_EVAL / "logs"

for p in [DATA_RAW, DATA_PROCESSED, FIG_DIR, TABLE_DIR, CACHE_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

WINDOWS_PATH = DATA_PROCESSED / "modserve_5min_windows.parquet"
METADATA_PATH = TABLE_DIR / "modserve_baseline_metadata.csv"

VAL_SUMMARY_PATH = TABLE_DIR / "modserve_val_baseline_summary.csv"
TEST_SUMMARY_PATH = TABLE_DIR / "modserve_test_baseline_summary.csv"
BURST_SUMMARY_PATH = TABLE_DIR / "modserve_burst_day_summary.csv"

print("Repo root:", REPO_ROOT)
print("Running in Colab:", IN_COLAB)
print("Windows path:", WINDOWS_PATH)

In [ ]:
# load the 5-minute dataset and metadata
if not WINDOWS_PATH.exists():
    raise FileNotFoundError(
        f"Missing 5-minute dataset: {WINDOWS_PATH}\n"
        "Run Notebook 2 first."
    )

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing metadata file: {METADATA_PATH}\n"
        "Run Notebook 2 first."
    )

windows = pd.read_parquet(WINDOWS_PATH)
windows["window_start"] = pd.to_datetime(windows["window_start"], utc=True)
windows["day"] = pd.to_datetime(windows["day"], utc=True)

metadata = pd.read_csv(METADATA_PATH)
metadata_dict = dict(zip(metadata["key"], metadata["value"]))

burst_day = pd.to_datetime(metadata_dict["burst_day"], utc=True)

windows.head()


In [ ]:
#quick sanity check
print("Number of windows:", len(windows))
print("Time range:", windows["window_start"].min(), "to", windows["window_start"].max())
print("\nSplit counts:")
print(windows["split"].value_counts())

print("\nAvailable columns:")
print(windows.columns.tolist())

In [ ]:
#define monolith evaluation function
def summarize_monolith(df, baseline_name, provision_col):
    tmp = df.copy()

    tmp["provided_total_units"] = tmp[provision_col]

    tmp["under_units"] = (tmp["req_total_units"] - tmp["provided_total_units"]).clip(lower=0)
    tmp["over_units"] = (tmp["provided_total_units"] - tmp["req_total_units"]).clip(lower=0)

    tmp["slo_safe_window"] = tmp["under_units"] == 0

    return {
        "baseline": baseline_name,
        "n_windows": len(tmp),
        "avg_required_units": tmp["req_total_units"].mean(),
        "avg_provisioned_units": tmp["provided_total_units"].mean(),
        "peak_required_units": tmp["req_total_units"].max(),
        "peak_provisioned_units": tmp["provided_total_units"].max(),
        "slo_safe_rate": tmp["slo_safe_window"].mean(),
        "underprovision_rate": (tmp["under_units"] > 0).mean(),
        "overprovision_rate": (tmp["over_units"] > 0).mean(),
        "avg_under_units": tmp["under_units"].mean(),
        "avg_over_units": tmp["over_units"].mean(),
    }

In [ ]:
#define decoupled evaluation function
def summarize_decoupled(df, baseline_name, image_col, text_col):
    tmp = df.copy()

    tmp["provided_image_units"] = tmp[image_col]
    tmp["provided_text_units"] = tmp[text_col]
    tmp["provided_total_units"] = tmp["provided_image_units"] + tmp["provided_text_units"]

    tmp["under_image_units"] = (tmp["req_image_units"] - tmp["provided_image_units"]).clip(lower=0)
    tmp["under_text_units"] = (tmp["req_text_units"] - tmp["provided_text_units"]).clip(lower=0)

    tmp["over_image_units"] = (tmp["provided_image_units"] - tmp["req_image_units"]).clip(lower=0)
    tmp["over_text_units"] = (tmp["provided_text_units"] - tmp["req_text_units"]).clip(lower=0)

    tmp["under_units"] = tmp["under_image_units"] + tmp["under_text_units"]
    tmp["over_units"] = tmp["over_image_units"] + tmp["over_text_units"]

    tmp["slo_safe_window"] = (tmp["under_image_units"] == 0) & (tmp["under_text_units"] == 0)

    return {
        "baseline": baseline_name,
        "n_windows": len(tmp),
        "avg_required_units": tmp["req_total_units"].mean(),
        "avg_provisioned_units": tmp["provided_total_units"].mean(),
        "avg_provisioned_image_units": tmp["provided_image_units"].mean(),
        "avg_provisioned_text_units": tmp["provided_text_units"].mean(),
        "peak_required_units": tmp["req_total_units"].max(),
        "peak_provisioned_units": tmp["provided_total_units"].max(),
        "slo_safe_rate": tmp["slo_safe_window"].mean(),
        "underprovision_rate": (tmp["under_units"] > 0).mean(),
        "overprovision_rate": (tmp["over_units"] > 0).mean(),
        "avg_under_units": tmp["under_units"].mean(),
        "avg_over_units": tmp["over_units"].mean(),
    }

In [ ]:
#split validation and test data
val_df = windows[windows["split"] == "val"].copy()
test_df = windows[windows["split"] == "test"].copy()

print("Validation windows:", len(val_df))
print("Test windows:", len(test_df))

In [ ]:
#evaluate baselines on validation split
val_results = []

val_results.append(
    summarize_monolith(
        val_df,
        baseline_name="Fixed Monolith",
        provision_col="prov_monolith_total_units"
    )
)

val_results.append(
    summarize_decoupled(
        val_df,
        baseline_name="Fixed Decoupled",
        image_col="prov_fixed_image_units",
        text_col="prov_fixed_text_units"
    )
)

val_results.append(
    summarize_decoupled(
        val_df,
        baseline_name="Reactive Decoupled",
        image_col="prov_reactive_image_units",
        text_col="prov_reactive_text_units"
    )
)

val_results_df = pd.DataFrame(val_results)
val_results_df

In [ ]:
#evaluate baselines on test split
test_results = []

test_results.append(
    summarize_monolith(
        test_df,
        baseline_name="Fixed Monolith",
        provision_col="prov_monolith_total_units"
    )
)

test_results.append(
    summarize_decoupled(
        test_df,
        baseline_name="Fixed Decoupled",
        image_col="prov_fixed_image_units",
        text_col="prov_fixed_text_units"
    )
)

test_results.append(
    summarize_decoupled(
        test_df,
        baseline_name="Reactive Decoupled",
        image_col="prov_reactive_image_units",
        text_col="prov_reactive_text_units"
    )
)

test_results_df = pd.DataFrame(test_results)
test_results_df

In [ ]:
# save validation and test summary tables
val_results_df.to_csv(VAL_SUMMARY_PATH, index=False)
test_results_df.to_csv(TEST_SUMMARY_PATH, index=False)

print("Saved:", VAL_SUMMARY_PATH)
print("Saved:", TEST_SUMMARY_PATH)

In [ ]:
#cleaner test summary view
display_cols = [
    "baseline",
    "avg_required_units",
    "avg_provisioned_units",
    "slo_safe_rate",
    "underprovision_rate",
    "overprovision_rate",
    "avg_under_units",
    "avg_over_units",
    "peak_required_units",
    "peak_provisioned_units",
]

test_results_df[display_cols].sort_values("slo_safe_rate", ascending=False)

In [ ]:
#plot test split comparison: SLO-safe and provision rates
plot_df = test_results_df.set_index("baseline")[
    ["slo_safe_rate", "underprovision_rate", "overprovision_rate"]
]

plot_df.plot(kind="bar", figsize=(10, 5))
plt.title("Test Split Baseline Comparison")
plt.xlabel("Baseline")
plt.ylabel("Rate")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIG_DIR / "modserve_test_baseline_rates.png", dpi=180)
plt.show()

In [ ]:
#plot average provisioned units
test_results_df.set_index("baseline")[["avg_provisioned_units"]].plot(kind="bar", figsize=(8, 5))
plt.title("Average Provisioned Units on Test Split")
plt.xlabel("Baseline")
plt.ylabel("Average Provisioned Units")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIG_DIR / "modserve_test_avg_provisioned_units.png", dpi=180)
plt.show()

In [ ]:
#isolate the chosen burst day
burst_df = test_df[test_df["day"] == burst_day].copy()

print("Chosen burst day:", burst_day)
print("Number of 5-minute windows on burst day:", len(burst_df))
burst_df.head()

In [ ]:
#plot the burst-day workload
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(burst_df["window_start"], burst_df["images_per_min"], label="Images / Min")
ax1.set_ylabel("Images / Min")
ax1.set_xlabel("Time")

ax2 = ax1.twinx()
ax2.plot(
    burst_df["window_start"],
    burst_df["context_tokens_per_min"],
    linestyle="--",
    label="Context Tokens / Min"
)
ax2.set_ylabel("Context Tokens / Min")

plt.title("Burst Day Workload")
fig.tight_layout()
plt.savefig(FIG_DIR / "modserve_burst_day_workload.png", dpi=180)
plt.show()

In [ ]:
#plot required vs provisioned units on burst day
burst_plot = burst_df[
    [
        "window_start",
        "req_total_units",
        "prov_monolith_total_units",
        "prov_fixed_decoupled_total_units",
        "prov_reactive_total_units",
    ]
].copy()

burst_plot = burst_plot.rename(columns={
    "req_total_units": "Required Units",
    "prov_monolith_total_units": "Fixed Monolith",
    "prov_fixed_decoupled_total_units": "Fixed Decoupled",
    "prov_reactive_total_units": "Reactive Decoupled",
})

burst_plot = burst_plot.set_index("window_start")

burst_plot.plot(figsize=(14, 5))
plt.title("Burst Day: Required vs Provisioned Units")
plt.xlabel("Time")
plt.ylabel("Units")
plt.tight_layout()
plt.savefig(FIG_DIR / "modserve_burst_day_required_vs_provisioned.png", dpi=180)
plt.show()

In [ ]:
#evaluate baselines only on the burst day
burst_results = []

burst_results.append(
    summarize_monolith(
        burst_df,
        baseline_name="Fixed Monolith",
        provision_col="prov_monolith_total_units"
    )
)

burst_results.append(
    summarize_decoupled(
        burst_df,
        baseline_name="Fixed Decoupled",
        image_col="prov_fixed_image_units",
        text_col="prov_fixed_text_units"
    )
)

burst_results.append(
    summarize_decoupled(
        burst_df,
        baseline_name="Reactive Decoupled",
        image_col="prov_reactive_image_units",
        text_col="prov_reactive_text_units"
    )
)

burst_results_df = pd.DataFrame(burst_results)
burst_results_df

In [ ]:
# save burst-day summary
burst_results_df.to_csv(BURST_SUMMARY_PATH, index=False)
print("Saved:", BURST_SUMMARY_PATH)

In [ ]:
# final summary printout
print("Main outputs created in this notebook:")
print("-", VAL_SUMMARY_PATH)
print("-", TEST_SUMMARY_PATH)
print("-", BURST_SUMMARY_PATH)
print("-", FIG_DIR / "modserve_test_baseline_rates.png")
print("-", FIG_DIR / "modserve_test_avg_provisioned_units.png")
print("-", FIG_DIR / "modserve_burst_day_workload.png")
print("-", FIG_DIR / "modserve_burst_day_required_vs_provisioned.png")